# Domain C: Contextual Adaptation and History Dependence

This notebook addresses three questions:
- **C1:** Do responses differ between the first and second presentation of the same block type (context-dependent adaptation)?
- **C2:** Does the context (contrast-varying vs. speed-varying) alter tuning curves?
- **C3:** Do responses change across days (multi-day representational drift)?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).
Context structure: Blocks 0,2 = contrast-context (TF fixed at 1 Hz); Blocks 1,3 = speed-context (contrast fixed at 0.8).

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, wilcoxon, mannwhitneyu, spearmanr, pearsonr, ttest_rel
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import get_block_responses, compute_adaptation_index, get_trial_position_responses, exp_decay, linear_CKA
from functions.tuning import naka_rushton, normalization_model

sns.set_context('talk')
sns.set_style('whitegrid')

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

print(f"Total cells: {len(obs)}, Total trials: {X.shape[1]}")

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

## C2: Context-Dependent Tuning Curve Modulation

Compare contrast response functions between contrast-context blocks (where contrast is variable) vs the single matched contrast available in speed-context blocks. Similarly for TF tuning. Test for gain control signatures.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C2.1  Compare CRF between contrast-context blocks and matched contrasts
#       in speed-context blocks
# ══════════════════════════════════════════════════════════════════════

# CRF from contrast-context (blocks 0+2 combined): full 5-level CRF
contrast_blocks = var['stim_block'].isin([0.0, 2.0])
crf_contrast_ctx = np.zeros((adata.n_obs, len(contrasts)))
for i, c in enumerate(contrasts):
    mask = contrast_blocks & (var['contrast'] == c)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 0:
        crf_contrast_ctx[:, i] = np.nanmean(X[:, trial_idx], axis=1)

# In speed-context blocks (1+3), contrast is fixed at 0.8
# So we get only a single point on the CRF from this context
speed_blocks = var['stim_block'].isin([1.0, 3.0])
mask_08_speed = speed_blocks & (var['contrast'] == 0.8)
resp_08_speed_ctx = np.nanmean(X[:, np.where(mask_08_speed.values)[0]], axis=1)

# Compare: response to contrast=0.8 in contrast-context vs speed-context
resp_08_contrast_ctx = crf_contrast_ctx[:, 4]  # contrast=0.8 is index 4

# ── TF tuning from speed-context (blocks 1+3): full 5-level TF curve ──
tf_speed_ctx = np.zeros((adata.n_obs, len(tfs)))
for i, tf in enumerate(tfs):
    mask = speed_blocks & (var['temporal_frequency'] == tf)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 0:
        tf_speed_ctx[:, i] = np.nanmean(X[:, trial_idx], axis=1)

# In contrast-context blocks (0+2), TF is fixed at 1 Hz
mask_tf1_contrast = contrast_blocks & (var['temporal_frequency'] == 1.0)
resp_tf1_contrast_ctx = np.nanmean(X[:, np.where(mask_tf1_contrast.values)[0]], axis=1)
resp_tf1_speed_ctx = tf_speed_ctx[:, 0]  # TF=1 is index 0

# ── Naka-Rushton fit to CRF per subclass (from contrast context) ──
fig, axes = plt.subplots(2, len(present_subclasses), figsize=(4*len(present_subclasses), 10))

# Row 1: CRF with overlaid speed-context matched point
for ax, sc in zip(axes[0], present_subclasses):
    mask = obs['subclass_name'].values == sc
    n_sc = mask.sum()
    
    # CRF from contrast context
    mean_crf = np.nanmean(crf_contrast_ctx[mask], axis=0)
    sem_crf = np.nanstd(crf_contrast_ctx[mask], axis=0) / np.sqrt(n_sc)
    ax.errorbar(contrasts, mean_crf, yerr=sem_crf, color=SUBCLASS_COLORS[sc],
                linewidth=2, capsize=3, marker='o', label='Contrast ctx')
    
    # Single point from speed context (c=0.8)
    mean_sp = np.nanmean(resp_08_speed_ctx[mask])
    sem_sp = np.nanstd(resp_08_speed_ctx[mask]) / np.sqrt(n_sc)
    ax.errorbar(0.8, mean_sp, yerr=sem_sp, marker='s', ms=10, color='red',
                zorder=5, label='Speed ctx (c=0.8)')
    
    ax.set_xscale('log')
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Contrast')
    if ax == axes[0, 0]:
        ax.set_ylabel('Mean ΔF/F')
        ax.legend(fontsize=7)

# Row 2: TF tuning with overlaid contrast-context matched point
for ax, sc in zip(axes[1], present_subclasses):
    mask = obs['subclass_name'].values == sc
    n_sc = mask.sum()
    
    mean_tf = np.nanmean(tf_speed_ctx[mask], axis=0)
    sem_tf = np.nanstd(tf_speed_ctx[mask], axis=0) / np.sqrt(n_sc)
    ax.errorbar(tfs, mean_tf, yerr=sem_tf, color=SUBCLASS_COLORS[sc],
                linewidth=2, capsize=3, marker='o', label='Speed ctx')
    
    mean_ct = np.nanmean(resp_tf1_contrast_ctx[mask])
    sem_ct = np.nanstd(resp_tf1_contrast_ctx[mask]) / np.sqrt(n_sc)
    ax.errorbar(1.0, mean_ct, yerr=sem_ct, marker='s', ms=10, color='blue',
                zorder=5, label='Contrast ctx (TF=1)')
    
    ax.set_xscale('log')
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Temporal Frequency (Hz)')
    if ax == axes[1, 0]:
        ax.set_ylabel('Mean ΔF/F')
        ax.legend(fontsize=7)

plt.suptitle('C2: Context-Dependent Tuning Curves\n(Top: CRF, Bottom: TF Tuning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Statistical test: paired context comparison at matched stimulus ──
print("=== Context Modulation: Contrast=0.8 response (contrast-ctx vs speed-ctx) ===")
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    v1 = resp_08_contrast_ctx[mask]
    v2 = resp_08_speed_ctx[mask]
    valid = ~np.isnan(v1) & ~np.isnan(v2)
    if valid.sum() >= 10:
        stat, p = wilcoxon(v1[valid], v2[valid])
        diff = np.mean(v1[valid]) - np.mean(v2[valid])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: Δ={diff:+.4f}, Wilcoxon p={p:.2e}")

print("\n=== Context Modulation: TF=1 response (speed-ctx vs contrast-ctx) ===")
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    v1 = resp_tf1_speed_ctx[mask]
    v2 = resp_tf1_contrast_ctx[mask]
    valid = ~np.isnan(v1) & ~np.isnan(v2)
    if valid.sum() >= 10:
        stat, p = wilcoxon(v1[valid], v2[valid])
        diff = np.mean(v1[valid]) - np.mean(v2[valid])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: Δ={diff:+.4f}, Wilcoxon p={p:.2e}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C2.2  Normalization / Gain Control Model Fit
#       Fit the Carandini-Heeger divisive normalization model to CRFs
#       Compare normalization pool parameters across cell types
# ══════════════════════════════════════════════════════════════════════

# Fit per cell for contrast-context CRFs
norm_params = {'C50_norm': [], 'Rmax_norm': [], 'n_norm': []}
for i in range(adata.n_obs):
    resp = crf_contrast_ctx[i]
    if np.ptp(resp) < 0.01 or np.any(np.isnan(resp)):
        norm_params['C50_norm'].append(np.nan)
        norm_params['Rmax_norm'].append(np.nan)
        norm_params['n_norm'].append(np.nan)
        continue
    try:
        popt, _ = curve_fit(normalization_model, contrasts, resp,
                           p0=[np.max(resp), 0.3, 2.0],
                           bounds=([0, 0.01, 0.1], [20, 2.0, 10]),
                           maxfev=5000)
        norm_params['C50_norm'].append(popt[1])
        norm_params['Rmax_norm'].append(popt[0])
        norm_params['n_norm'].append(popt[2])
    except:
        norm_params['C50_norm'].append(np.nan)
        norm_params['Rmax_norm'].append(np.nan)
        norm_params['n_norm'].append(np.nan)

adapt_df['C50_norm'] = norm_params['C50_norm']
adapt_df['Rmax_norm'] = norm_params['Rmax_norm']
adapt_df['n_norm'] = norm_params['n_norm']

# ── Visualization: Normalization parameters by subclass ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, param in zip(axes, ['C50_norm', 'Rmax_norm', 'n_norm']):
    sns.violinplot(data=adapt_df[adapt_df['subclass'].isin(present_subclasses)],
                   x='subclass_short', y=param, order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.set_title(f'C2: {param}', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('C2: Normalization Model Parameters by Subclass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()